# JFCNN Training — Productive vs Waste

Trains the Joint Semantic + Structural CNN on the labeled page corpus.

**Pipeline**
1. Load labeled records from SQLite, drop `skip`
2. Build vocab + GloVe / structural embedding matrices
3. Stratified 80/20 train/val split
4. Train with Adam + CrossEntropyLoss
5. Evaluate: accuracy, precision, recall, F1, AUC
6. Save model weights + vocab artifacts to `checkpoints/`

> **Note — small dataset**: after dropping `skip`, the corpus currently has ~19 pages.
> All metrics on a 4-sample val set are high-variance. Treat them as sanity checks
> until the corpus grows.

In [ ]:
import sys
sys.path.insert(0, '.')

import json
import os
import pickle
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

import config
from semantic_structure.db import load_records
from semantic_structure.extractor import build_corpus_vocab
from semantic_structure.dataset import PageDataset
from semantic_structure.model import JFCNN

print('Imports OK')

## Hyperparameters

In [ ]:
# ── Data ───────────────────────────────────────────────────────────────────
VAL_SPLIT    = 0.20      # fraction of data held out for validation
SEED         = 42

# ── Model ──────────────────────────────────────────────────────────────────
NUM_FILTERS  = 128       # conv filters per kernel size
KERNEL_SIZES = [3, 4, 5] # parallel conv towers
FC_HIDDEN    = 256
DROPOUT      = 0.5
# TODO: set FREEZE_WORD_EMB=False to fine-tune GloVe weights once the
#       corpus is large enough (>1 k labeled pages recommended).
FREEZE_WORD_EMB = True

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE   = 8
NUM_EPOCHS   = 30
LR           = 1e-3

# ── Paths ──────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'models', 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device:          {DEVICE}')
print(f'Checkpoint dir:  {CHECKPOINT_DIR}')

## 1. Load data + build vocab

In [ ]:
from collections import Counter

all_records = load_records(config.DB_PATH)
print(f'Total labeled records: {len(all_records)}')
print(f'Label breakdown: {dict(Counter(r[2] for r in all_records))}')

# build_corpus_vocab uses ALL records (including skip) for vocabulary coverage,
# then PageDataset drops skip rows at construction time.
token2idx, word_matrix, struct_matrix = build_corpus_vocab(
    all_records, config.GLOVE_PATH, config.N, config.M
)

k = word_matrix.shape[1]
print(f'\nVocab size:  {len(token2idx)}')
print(f'k (GloVe):   {k}')
print(f'n (struct):  {config.N}')
print(f'Input dim:   {k + config.N}  (k+n per token)')

## 2. Dataset + stratified train/val split

In [ ]:
dataset = PageDataset(
    records   = all_records,
    token2idx = token2idx,
    tag2idx   = config.TAG_TO_IDX,
    label2idx = config.LABEL_TO_IDX,
    m         = config.M,
)

print(f'Dataset size after dropping skip: {len(dataset)}')
label_counts = Counter(dataset.records[i][2] for i in range(len(dataset)))
print(f'Label breakdown: {dict(label_counts)}')

# Stratified split: keep class ratio in both train and val
rng = random.Random(SEED)

by_class = {}
for i, (_, _, label_str) in enumerate(dataset.records):
    by_class.setdefault(label_str, []).append(i)

train_indices, val_indices = [], []
for label_str, indices in by_class.items():
    rng.shuffle(indices)
    n_val = max(1, round(len(indices) * VAL_SPLIT))
    val_indices.extend(indices[:n_val])
    train_indices.extend(indices[n_val:])

rng.shuffle(train_indices)
rng.shuffle(val_indices)

train_set = Subset(dataset, train_indices)
val_set   = Subset(dataset, val_indices)

print(f'\nTrain: {len(train_set)} samples')
print(f'Val:   {len(val_set)} samples')

In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')

## 3. Build model

In [ ]:
torch.manual_seed(SEED)

model = JFCNN(
    word_matrix    = word_matrix,
    struct_matrix  = struct_matrix,
    num_filters    = NUM_FILTERS,
    kernel_sizes   = KERNEL_SIZES,
    fc_hidden      = FC_HIDDEN,
    dropout        = DROPOUT,
    num_classes    = 2,
    freeze_word_emb = FREEZE_WORD_EMB,
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal params:     {total:,}')
print(f'Trainable params: {trainable:,}')

## 4. Optimizer + loss

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()  # applies log-softmax internally

## 5. Training loop

In [ ]:
def evaluate(model, loader, criterion, device):
    """Run one pass over loader, return loss + all metrics."""
    model.eval()
    all_labels = []
    all_preds  = []
    all_probs  = []   # P(waste) for AUC
    total_loss = 0.0

    with torch.no_grad():
        for word_idx, tag_idx, labels in loader:
            word_idx = word_idx.to(device)
            tag_idx  = tag_idx.to(device)
            labels   = labels.to(device)

            logits = model(word_idx, tag_idx)
            loss   = criterion(logits, labels)
            total_loss += loss.item() * len(labels)

            probs = torch.softmax(logits, dim=-1)[:, 1]  # P(waste)
            preds = logits.argmax(dim=-1)

            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    n = len(all_labels)
    avg_loss  = total_loss / n
    accuracy  = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)
    f1        = f1_score(all_labels, all_preds, zero_division=0)
    # AUC requires at least one sample from each class in the split
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')

    return dict(loss=avg_loss, accuracy=accuracy,
                precision=precision, recall=recall, f1=f1, auc=auc)


history = []

print(f'{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>8}  {'Acc':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  {'AUC':>6}')
print('-' * 75)

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for word_idx, tag_idx, labels in train_loader:
        word_idx = word_idx.to(DEVICE)
        tag_idx  = tag_idx.to(DEVICE)
        labels   = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(word_idx, tag_idx)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)

    train_loss /= len(train_set)

    # ── Validate ───────────────────────────────────────────────────────────
    val_metrics = evaluate(model, val_loader, criterion, DEVICE)
    history.append({'epoch': epoch, 'train_loss': train_loss, **val_metrics})

    if epoch % 5 == 0 or epoch == 1:
        print(
            f'{epoch:>6}  {train_loss:>10.4f}  '
            f'{val_metrics["loss"]:>8.4f}  '
            f'{val_metrics["accuracy"]:>6.3f}  '
            f'{val_metrics["precision"]:>6.3f}  '
            f'{val_metrics["recall"]:>6.3f}  '
            f'{val_metrics["f1"]:>6.3f}  '
            f'{val_metrics["auc"]:>6.3f}'
        )

print('\nTraining complete.')

## 6. Final evaluation

In [ ]:
final = evaluate(model, val_loader, criterion, DEVICE)

print('Val metrics (final epoch)')
print('-' * 30)
for name, val in final.items():
    print(f'  {name:12s}: {val:.4f}')

## 7. Save model + vocab artifacts

In [ ]:
# Model weights
model_path = os.path.join(CHECKPOINT_DIR, 'jfcnn.pt')
torch.save(model.state_dict(), model_path)
print(f'Model weights saved → {model_path}')

# token2idx (needed to tokenise new pages at inference time)
vocab_path = os.path.join(CHECKPOINT_DIR, 'token2idx.json')
with open(vocab_path, 'w') as f:
    json.dump(token2idx, f)
print(f'token2idx saved    → {vocab_path}')

# Embedding matrices (word_matrix is large; struct_matrix is trained)
emb_path = os.path.join(CHECKPOINT_DIR, 'embeddings.npz')
np.savez(emb_path, word_matrix=word_matrix, struct_matrix=struct_matrix)
print(f'Embeddings saved   → {emb_path}')

# Training config for reproducibility
cfg_path = os.path.join(CHECKPOINT_DIR, 'train_config.json')
with open(cfg_path, 'w') as f:
    json.dump(dict(
        num_filters=NUM_FILTERS,
        kernel_sizes=KERNEL_SIZES,
        fc_hidden=FC_HIDDEN,
        dropout=DROPOUT,
        freeze_word_emb=FREEZE_WORD_EMB,
        num_classes=2,
        batch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        lr=LR,
        seed=SEED,
        val_split=VAL_SPLIT,
        final_val_metrics=final,
    ), f, indent=2)
print(f'Train config saved → {cfg_path}')